In [0]:
df = spark.readStream.format('cloudFiles')\
            .option('cloudFiles.format', 'json')\
            .option('cloudFiles.inferColumnTypes', 'true')\
            .option('cloudFiles.schemaLocation', '/Volumes/fraud_detection/source/schema/fraud_watchlist/')\
            .load('/Volumes/fraud_detection/source/fraud_watchlist/fraud/')

In [0]:
from pyspark.sql.functions import col, current_timestamp
df = df.select('*',
            col('_metadata.file_path').alias('file_path'),
            col('_metadata.file_modification_time').alias('file_modification_time'),
            current_timestamp().alias('ingest_time'))

In [0]:
df.writeStream\
    .format('delta')\
    .outputMode('append')\
    .option('checkpointLocation', '/Volumes/fraud_detection/source/checkpointlocation/fraud_watchlist/')\
    .trigger(availableNow = True)\
    .toTable('fraud_detection.bronze.fraud_watchlist')\
    .awaitTermination()

In [0]:
%sql
select * from fraud_detection.bronze.fraud_watchlist